# FinSight – Financial Statement & Budget Variance Analysis
**Project:** Financial Analytics and Risk Reporting System (Nexvia Internship Evaluation)
**Objective:** Ingest double-entry transaction ledgers, calculate GAAP P&L metrics, verify Balance Sheet reconciliation, and identify budget variances.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Connect to FinSight SQLite Database
conn = sqlite3.connect('../finsight.db')
print('Database connected successfully.')

## 1. Monthly Profit & Loss Statement
Extract calculated revenue, COGS, EBITDA, and net profit from SQL reporting views.

In [2]:
query_pnl = 'SELECT * FROM view_monthly_pnl ORDER BY fiscal_month'
df_pnl = pd.read_sql(query_pnl, conn)
df_pnl['gross_margin_%'] = (df_pnl['gross_profit'] / df_pnl['total_revenue'] * 100).round(2)
df_pnl['net_margin_%'] = (df_pnl['net_profit'] / df_pnl['total_revenue'] * 100).round(2)
display(df_pnl[['fiscal_month', 'total_revenue', 'cogs', 'gross_profit', 'ebitda', 'net_profit', 'gross_margin_%', 'net_margin_%']])

## 2. Balance Sheet Reconciliation Verification
Verify that Assets = Liabilities + Equity for all 12 fiscal months.

In [3]:
query_bs = '''
    SELECT 
        a.fiscal_month,
        ROUND(SUM(a.net_value), 2) AS total_assets,
        ROUND((SELECT SUM(l.current_balance) FROM liabilities l WHERE l.fiscal_month = a.fiscal_month), 2) AS total_liabilities,
        ROUND((SELECT SUM(e.total_equity) FROM equity e WHERE e.fiscal_month = a.fiscal_month), 2) AS total_equity
    FROM assets a
    GROUP BY a.fiscal_month
'''
df_bs = pd.read_sql(query_bs, conn)
df_bs['discrepancy'] = (df_bs['total_assets'] - (df_bs['total_liabilities'] + df_bs['total_equity'])).round(2)
df_bs['is_reconciled'] = df_bs['discrepancy'].abs() < 0.01
display(df_bs)
print('All months reconciled:', df_bs['is_reconciled'].all())

## 3. Department Budget vs Actual Variance
Analyze which departments are exceeding budgets and driving financial variances.

In [4]:
query_bva = '''
    SELECT 
        department_name,
        ROUND(SUM(budgeted_amount), 2) AS total_budget,
        ROUND(SUM(actual_amount), 2) AS total_actual,
        ROUND(SUM(actual_amount) - SUM(budgeted_amount), 2) AS variance_amount,
        ROUND(((SUM(actual_amount) - SUM(budgeted_amount)) / NULLIF(SUM(budgeted_amount), 0)) * 100.0, 2) AS variance_pct
    FROM view_budget_vs_actual
    WHERE account_type = 'Expense'
    GROUP BY department_name
    ORDER BY variance_amount DESC
'''
df_bva = pd.read_sql(query_bva, conn)
display(df_bva)